# Creazione di applicazioni per la generazione di immagini (OpenAI)

Gli LLM non servono solo per generare testo. Puoi anche generare immagini a partire da descrizioni testuali. Le immagini come modalità sono utili in MedTech, architettura, turismo, sviluppo di giochi, marketing e altro ancora. In questa lezione lavoriamo con i modelli **GPT Image** di oggi sulla piattaforma OpenAI.

## Obiettivi di apprendimento

Alla fine di questa lezione sarai in grado di:

- Spiegare cos’è la generazione di immagini e dove è utile.
- Comprendere la famiglia di modelli `gpt-image` e come si differenzia dai modelli legacy DALL·E.
- Creare un’applicazione di generazione di immagini e modificare immagini.

## Cos’è la generazione di immagini?

I modelli di generazione di immagini creano immagini a partire da un prompt testuale. I modelli moderni come `gpt-image` apprendono durante l’addestramento la relazione tra testo e immagini, quindi trasformano iterativamente del rumore casuale in un’immagine che corrisponde al tuo prompt.

Due famiglie di modelli di immagini ben conosciute sono:

- **`gpt-image` (OpenAI)** — la generazione attuale usata in questa lezione. Supporta la generazione testo-immagine e la modifica delle immagini (inpainting con una maschera).
- **Midjourney** — un modello di terze parti popolare con un proprio servizio e workflow basato su Discord.

> I modelli di immagini OpenAI più vecchi — **DALL·E 2** e **DALL·E 3** — sono legacy. DALL·E 3 non è più disponibile per nuove implementazioni, e feature come `create_variation` esistevano solo in DALL·E 2. Usa i modelli `gpt-image` per nuove applicazioni.

> **Importante:** i modelli `gpt-image` restituiscono l’immagine generata come **base64** (`b64_json`), non come URL. Il tuo codice decodifica la stringa base64 in byte e la salva — non c’è un URL immagine da scaricare.


## Costruire la tua prima applicazione di generazione di immagini

Quindi, cosa serve per costruire un'applicazione di generazione di immagini? Ti servono le seguenti librerie:

- **python-dotenv**, è fortemente consigliato usare questa libreria per conservare i tuoi segreti in un file *.env* separato dal codice.
- **openai**, questa libreria è quella che userai per interagire con l'API di OpenAI.
- **pillow**, per lavorare con le immagini in Python.


1. Crea un file *.env* con il seguente contenuto:

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. Raccogli le librerie sopra in un file chiamato *requirements.txt* come segue:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Successivamente, crea un ambiente virtuale e installa le librerie:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Per Windows, usa i seguenti comandi per creare e attivare il tuo ambiente virtuale:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Aggiungi il seguente codice in un file chiamato *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # import dotenv
    dotenv.load_dotenv()

    # Crea l'oggetto OpenAI (legge OPENAI_API_KEY dal tuo .env)
    client = openai.OpenAI()


    try:
        # Crea un'immagine usando l'API di generazione immagini
        generation_response = client.images.generate(
            model="gpt-image-1",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Inserisci qui il testo del tuo prompt
            size='1024x1024',
            n=1
        )
        # Imposta la directory per l'immagine salvata
        image_dir = os.path.join(os.curdir, 'images')

        # Se la directory non esiste, creala
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Inizializza il percorso dell'immagine (nota che il formato file dovrebbe essere png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # I modelli gpt-image restituiscono l'immagine come base64 (b64_json), non un URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Visualizza l'immagine nel visualizzatore immagini predefinito
        image = Image.open(image_path)
        image.show()

    # cattura le eccezioni
    except openai.BadRequestError as err:
        print(err)

    ```

Spieghiamo questo codice:

- Prima, importiamo le librerie di cui abbiamo bisogno, inclusa la libreria OpenAI, la libreria dotenv, il modulo base64 e la libreria Pillow.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- Dopo di che, creiamo il client, che legge la chiave API dal tuo file ``.env``.

    ```python
    # Crea oggetto OpenAI
    client = openai.OpenAI()
    ```

- Successivamente, generiamo l'immagine:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-1",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Inserisci qui il testo del tuo prompt
        size='1024x1024',
        n=1
    )
    ```

    I modelli `gpt-image` restituiscono l'immagine come una stringa **base64** in `data[0].b64_json`. La decodifichiamo in byte e la scriviamo su un file — non c'è alcun URL da scaricare.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Infine, apriamo l'immagine e utilizziamo il visualizzatore standard per mostrarla:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Più dettagli sulla generazione dell'immagine

Esaminiamo i parametri di `images.generate`:

- **model**, è il modello di immagine da usare (per esempio `gpt-image-1`).
- **prompt**, è il testo usato per generare l'immagine. Qui è "Coniglio su un cavallo, che tiene un lecca-lecca, in un prato nebbioso dove crescono narcisi".
- **size**, è la dimensione dell'immagine generata (per esempio `1024x1024`, `1536x1024`, `1024x1536`, o `"auto"`).
- **n**, è il numero di immagini generate. Qui ne generiamo una.

> I modelli di immagini non accettano un parametro `temperature` — quello è un controllo per la generazione di testo. Per ottenere varietà, chiama l'API di nuovo; per ridurre la varietà, rendi il tuo prompt più specifico.

## Capacità aggiuntive della generazione di immagini

Hai visto come generare un'immagine con poche righe di Python. I modelli `gpt-image` possono anche **modificare** un'immagine esistente. Fornendo un'immagine, una **maschera** opzionale (che segna l'area da modificare) e un prompt, puoi alterare una parte di un'immagine — per esempio, aggiungere un cappello al nostro coniglio.

```python
response = client.images.edit(
  model="gpt-image-1",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# le modifiche sono restituite anche come base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

L'immagine di base contiene solo il coniglio; l'immagine finale ha il cappello sul coniglio.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Questo documento è stato tradotto utilizzando il servizio di traduzione AI [Co-op Translator](https://github.com/Azure/co-op-translator). Sebbene ci impegniamo per garantire la precisione, si prega di notare che le traduzioni automatizzate possono contenere errori o imprecisioni. Il documento originale nella sua lingua nativa deve essere considerato la fonte autorevole. Per informazioni critiche, si raccomanda una traduzione professionale effettuata da un essere umano. Non siamo responsabili per eventuali malintesi o interpretazioni errate derivanti dall’uso di questa traduzione.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
